固定条件: model、revision、seed=42、prompt、max_new_tokens=64、CPU
唯一变量: do_sample
- do_sample=False: 贪心解码，输出稳定可复现
- do_sample=True: 随机采样，输出带有随机性，结果不固定

In [1]:
import transformers, torch
print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)

C:\Users\Administrator\.conda\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers version: 5.23.0
torch version: 2.15.0+cpu


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

cache_path = "./flan-t5-small-local"

tokenizer = AutoTokenizer.from_pretrained(
    cache_path,
    local_files_only=True
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    cache_path,
    local_files_only=True
)
model = model.to("cpu")

def infer(text, do_sample=False):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=do_sample
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

#正常样例
prompt1 = ("What is machine learning?")
out1_base = infer(prompt1, do_sample=False)   # 基准A：贪心
out1_ctl = infer(prompt1, do_sample=True)    # 对照B：仅改do_sample
print("输入1：",prompt1)
print("【基准 do_sample=False】输出：", out1_base)
print("【对照 do_sample=True】输出：", out1_ctl)
print("-"*50)

prompt2 = "Explain what is AI in one sentence."
out2_base = infer(prompt2, do_sample=False)
out2_ctl = infer(prompt2, do_sample=True)
print("输入2：", prompt2)
print("【基准 do_sample=False】输出：", out2_base)
print("【对照 do_sample=True】输出：", out2_ctl)
print("-"*50)

prompt3 = "Summarize: Apples are red and grow on trees."
out3_base = infer(prompt3, do_sample=False)
out3_ctl = infer(prompt3, do_sample=True)
print("输入3：", prompt3)
print("【基准 do_sample=False】输出：", out3_base)
print("【对照 do_sample=True】输出：", out3_ctl)
print("-"*50)

# 失败样例
bad_input = "aaaaa" * 1000
out4_base = infer(bad_input, do_sample=False)
out4_ctl = infer(bad_input, do_sample=True)
print("输入4（失败样例）：", bad_input[:200] + "...")
print("【基准 do_sample=False】输出：", out4_base)
print("【对照 do_sample=True】输出：", out4_ctl)
print("-"*50)

Loading weights: 100%|██████████| 192/192 [00:00<00:00, 17452.41it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


输入1： What is machine learning?
【基准 do_sample=False】输出： machine learning
【对照 do_sample=True】输出： machine learning () is a technique used in machine science to identify and formulate patterns of behaviour. This concept is also recognized as a subject of attention to detail and knowledge.
--------------------------------------------------
输入2： Explain what is AI in one sentence.
【基准 do_sample=False】输出： AI (human) is a cellular phone used to communicate with other humans.
【对照 do_sample=True】输出： AI is a computer model used to compute data and calculate the size of a frame, thus measuring the size of an AI.
--------------------------------------------------
输入3： Summarize: Apples are red and grow on trees.
【基准 do_sample=False】输出： Leave the apple in the tree for a few days.
【对照 do_sample=True】输出： Place an apple in the pear grate and lay it on the tree.
--------------------------------------------------
输入4（失败样例）： aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa

In [3]:
import json

def save_prediction(type_, prompt, do_sample, output):
    record = {
        "type": type_,
        "prompt": prompt,
        "generation_config": {
            "max_new_tokens": 64,
            "do_sample": do_sample
        },
        "raw_output": output
    }
    with open("predictions.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# 跑完一组就调用，示例：
save_prediction("normal", prompt1, False, out1_base)
save_prediction("normal", prompt1, True, out1_ctl)
save_prediction("normal", prompt2, False, out2_base)
save_prediction("normal", prompt2, True, out2_ctl)
save_prediction("normal", prompt3, False, out3_base)
save_prediction("normal", prompt3, True, out3_ctl)
save_prediction("failure", bad_input[:200]+"...", False, out4_base)
save_prediction("failure", bad_input[:200]+"...", True, out4_ctl)